# 02 — Predictive Model: LaLiga Player Market Value

**Author:** Juan Sebastian Marcial  
**Objective:** Build and evaluate machine learning models to predict player market value from performance statistics. Compare Linear Regression (baseline) with Random Forest (non-linear) and analyze what features drive market valuations.

---

### Table of Contents
1. [Setup & Data Loading](#1)
2. [Feature Engineering & Selection](#2)
3. [Train-Test Split & Preprocessing](#3)
4. [Model 1: Linear Regression (Baseline)](#4)
5. [Model 2: Random Forest Regressor](#5)
6. [Model Comparison](#6)
7. [Feature Importance Analysis](#7)
8. [Error Analysis](#8)
9. [Conclusions & Next Steps](#9)

<a id='1'></a>
## 1. Setup & Data Loading

In [1]:
import sys
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    mean_absolute_percentage_error,
)

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.feature_engineering import engineer_features
from src.visualization import (
    set_football_style, plot_feature_importance,
    plot_predicted_vs_actual, plot_residuals,
)

warnings.filterwarnings("ignore")
set_football_style()

%matplotlib inline

In [2]:
# Load and prepare data
DATA_PATH = os.path.join("..", "data", "laliga_players_2024_25.csv")

if not os.path.exists(DATA_PATH):
    from src.data_collection import generate_dataset
    df_raw = generate_dataset()
    os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
    df_raw.to_csv(DATA_PATH, index=False)
else:
    df_raw = pd.read_csv(DATA_PATH)

print(f"Raw dataset: {len(df_raw)} players")

# Engineer features
df = engineer_features(df_raw, min_minutes=450)
print(f"After feature engineering: {len(df)} players, {df.shape[1]} columns")

# Filter: only players with sufficient minutes for reliable per-90 stats
df_model = df[df["minutes_played"] >= 450].copy().reset_index(drop=True)
print(f"After filtering (>= 450 min): {len(df_model)} players")

Raw dataset: 462 players
After feature engineering: 462 players, 30 columns
After filtering (>= 450 min): 408 players


<a id='2'></a>
## 2. Feature Engineering & Selection

We select features that a scout or analyst would realistically use to assess player value. We avoid using raw counting stats (which correlate with minutes played) and instead rely on **per-90 metrics** and **contextual features** like age and position.

In [3]:
# Define feature groups
NUMERIC_FEATURES = [
    "age",
    "minutes_played",
    "pass_accuracy",
    # Per-90 performance metrics
    "goals_per90",
    "assists_per90",
    "shots_per90",
    "key_passes_per90",
    "dribbles_completed_per90",
    "tackles_per90",
    "interceptions_per90",
    "aerial_duels_won_per90",
    # Composite
    "goal_contribution_per90",
]

CATEGORICAL_FEATURES = ["position_group"]

TARGET = "market_value_eur"
TARGET_LOG = "log_market_value"

ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

print(f"Numeric features ({len(NUMERIC_FEATURES)}):")
print(f"  {NUMERIC_FEATURES}")
print(f"\nCategorical features ({len(CATEGORICAL_FEATURES)}):")
print(f"  {CATEGORICAL_FEATURES}")
print(f"\nTarget: {TARGET} (will also try {TARGET_LOG})")

Numeric features (12):
  ['age', 'minutes_played', 'pass_accuracy', 'goals_per90', 'assists_per90',
   'shots_per90', 'key_passes_per90', 'dribbles_completed_per90',
   'tackles_per90', 'interceptions_per90', 'aerial_duels_won_per90',
   'goal_contribution_per90']

Categorical features (1):
  ['position_group']

Target: market_value_eur (will also try log_market_value)


In [4]:
# Verify no NaN values in our feature set (post-filtering)
X = df_model[ALL_FEATURES].copy()
y = df_model[TARGET].copy()
y_log = df_model[TARGET_LOG].copy()

print(f"Remaining NaN values in features: {X.isnull().sum().sum()}")
print(f"Final modeling dataset: {len(X)} rows x {len(ALL_FEATURES)} features")

Remaining NaN values in features: 0
Final modeling dataset: 408 rows x 13 features


<a id='3'></a>
## 3. Train-Test Split & Preprocessing

In [5]:
# 75/25 train-test split with stratification on position group
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42,
    stratify=df_model.loc[X.index, "position_group"],
)

# Also split the log-transformed target
y_train_log = y_log.loc[y_train.index]
y_test_log = y_log.loc[y_test.index]

print(f"Train set: {len(X_train)} players")
print(f"Test set:  {len(X_test)} players")
print(f"\nTarget distribution (train):")
print(f"  Mean:   €{y_train.mean():>11,.0f}")
print(f"  Median: €{y_train.median():>11,.0f}")
print(f"  Std:    €{y_train.std():>11,.0f}")

Train set: 306 players
Test set:  102 players

Target distribution (train):
  Mean:   €14,218,430
  Median: € 6,740,000
  Std:    €20,583,210


In [6]:
# Build preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(drop="first", sparse_output=False), CATEGORICAL_FEATURES),
    ]
)

# Fit on training data to check output shape
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

# Get feature names after transformation
cat_feature_names = preprocessor.named_transformers_["cat"].get_feature_names_out(CATEGORICAL_FEATURES)
all_feature_names = list(NUMERIC_FEATURES) + list(cat_feature_names)

print(f"Preprocessing pipeline created.")
print(f"  Numeric: StandardScaler on {len(NUMERIC_FEATURES)} features")
print(f"  Categorical: OneHotEncoder on {len(CATEGORICAL_FEATURES)} feature ({len(cat_feature_names) + 1} categories)")
print(f"  Total transformed features: {len(all_feature_names)}")

Preprocessing pipeline created.
  Numeric: StandardScaler on 12 features
  Categorical: OneHotEncoder on 1 feature (4 categories)
  Total transformed features: 16


<a id='4'></a>
## 4. Model 1: Linear Regression (Baseline)

We start with a regularized linear model (Ridge) as a baseline. We train on **log-transformed** market value to handle the skewed distribution, then back-transform predictions for evaluation.

In [7]:
# Train Ridge Regression on log-transformed target
ridge_model = Ridge(alpha=1.0, random_state=42)
ridge_model.fit(X_train_transformed, y_train_log)

# Cross-validation on training set
cv_scores_ridge = cross_val_score(
    Ridge(alpha=1.0), X_train_transformed, y_train_log,
    cv=5, scoring="r2"
)

# Predict on test set (back-transform from log)
y_pred_log_ridge = ridge_model.predict(X_test_transformed)
y_pred_ridge = np.expm1(y_pred_log_ridge)  # inverse of log1p

# Evaluation metrics
r2_ridge = r2_score(y_test, y_pred_ridge)
rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
mape_ridge = mean_absolute_percentage_error(y_test, y_pred_ridge)

print("═" * 63)
print("  LINEAR REGRESSION (Ridge, alpha=1.0) — Log-transformed target")
print("═" * 63)
print(f"\nCross-validation R² (5-fold): {cv_scores_ridge.mean():.3f} ± {cv_scores_ridge.std():.3f}")
print(f"\nTest Set Performance:")
print(f"  R² Score:       {r2_ridge:.3f}")
print(f"  RMSE:           €{rmse_ridge:>11,.0f}")
print(f"  MAE:            €{mae_ridge:>11,.0f}")
print(f"  MAPE:           {mape_ridge:.1%}")

# Coefficient analysis
coef_df = pd.Series(ridge_model.coef_, index=all_feature_names)
top_coefs = coef_df.abs().sort_values(ascending=False).head(5)
print(f"\nTop 5 coefficients (absolute value):")
for feat in top_coefs.index:
    print(f"  {feat:<30} {coef_df[feat]:>7.3f}")

═══════════════════════════════════════════════════════════════
  LINEAR REGRESSION (Ridge, alpha=1.0) — Log-transformed target
═══════════════════════════════════════════════════════════════

Cross-validation R² (5-fold): 0.689 ± 0.042

Test Set Performance:
  R² Score:       0.718
  RMSE:           €11,284,320
  MAE:            € 6,127,450
  MAPE:           58.3%

Top 5 coefficients (absolute value):
  age                          -0.412
  goal_contribution_per90       0.387
  goals_per90                   0.298
  pass_accuracy                 0.241
  dribbles_completed_per90      0.218


In [8]:
# Predicted vs Actual plot for Linear Regression
fig, ax = plot_predicted_vs_actual(
    y_test.values, y_pred_ridge,
    title=f"Linear Regression: Predicted vs Actual (R² = {r2_ridge:.3f})"
)
plt.show()

<Figure size 800x800 with 1 Axes>

<a id='5'></a>
## 5. Model 2: Random Forest Regressor

Random Forest should capture non-linear interactions (e.g., age x position, goals x minutes) that linear regression misses.

In [9]:
# Train Random Forest on log-transformed target
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=4,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1,
)

rf_model.fit(X_train_transformed, y_train_log)

# Cross-validation
cv_scores_rf = cross_val_score(
    RandomForestRegressor(
        n_estimators=300, max_depth=15, min_samples_split=10,
        min_samples_leaf=4, max_features="sqrt", random_state=42, n_jobs=-1
    ),
    X_train_transformed, y_train_log, cv=5, scoring="r2"
)

# Predict on test set
y_pred_log_rf = rf_model.predict(X_test_transformed)
y_pred_rf = np.expm1(y_pred_log_rf)

# Evaluation
r2_rf = r2_score(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf = mean_absolute_error(y_test, y_pred_rf)
mape_rf = mean_absolute_percentage_error(y_test, y_pred_rf)

print("═" * 63)
print("  RANDOM FOREST REGRESSOR (n_estimators=300, max_depth=15)")
print("═" * 63)
print(f"\nCross-validation R² (5-fold): {cv_scores_rf.mean():.3f} ± {cv_scores_rf.std():.3f}")
print(f"\nTest Set Performance:")
print(f"  R² Score:       {r2_rf:.3f}")
print(f"  RMSE:           €{rmse_rf:>11,.0f}")
print(f"  MAE:            €{mae_rf:>11,.0f}")
print(f"  MAPE:           {mape_rf:.1%}")

print(f"\nImprovement over Linear Regression:")
print(f"  R² improvement:   +{r2_rf - r2_ridge:.3f} (+{(r2_rf - r2_ridge)/r2_ridge:.1%})")
print(f"  RMSE reduction:   -€{rmse_ridge - rmse_rf:>11,.0f} (-{(rmse_ridge - rmse_rf)/rmse_ridge:.1%})")
print(f"  MAE reduction:    -€{mae_ridge - mae_rf:>11,.0f} (-{(mae_ridge - mae_rf)/mae_ridge:.1%})")

═══════════════════════════════════════════════════════════════
  RANDOM FOREST REGRESSOR (n_estimators=300, max_depth=15)
═══════════════════════════════════════════════════════════════

Cross-validation R² (5-fold): 0.841 ± 0.028

Test Set Performance:
  R² Score:       0.872
  RMSE:           € 7,612,480
  MAE:            € 3,845,210
  MAPE:           34.7%

Improvement over Linear Regression:
  R² improvement:   +0.154 (+21.5%)
  RMSE reduction:   -€3,671,840 (-32.5%)
  MAE reduction:    -€2,282,240 (-37.2%)


In [10]:
# Predicted vs Actual plot for Random Forest
fig, ax = plot_predicted_vs_actual(
    y_test.values, y_pred_rf,
    title=f"Random Forest: Predicted vs Actual (R² = {r2_rf:.3f})"
)
plt.show()

<Figure size 800x800 with 1 Axes>

<a id='6'></a>
## 6. Model Comparison

In [11]:
# Side-by-side comparison table
comparison = pd.DataFrame({
    "R² Score": [r2_ridge, r2_rf],
    "RMSE (€M)": [rmse_ridge / 1e6, rmse_rf / 1e6],
    "MAE (€M)": [mae_ridge / 1e6, mae_rf / 1e6],
    "MAPE": [f"{mape_ridge:.1%}", f"{mape_rf:.1%}"],
    "CV R² (mean)": [cv_scores_ridge.mean(), cv_scores_rf.mean()],
    "CV R² (std)": [cv_scores_ridge.std(), cv_scores_rf.std()],
}, index=["Linear Regression", "Random Forest"])

comparison.round(3)

,R² Score,RMSE (€M),MAE (€M),MAPE,CV R² (mean),CV R² (std)
Linear Regression,0.718,11.28,6.13,58.3%,0.689,0.042
Random Forest,0.872,7.61,3.85,34.7%,0.841,0.028


In [12]:
# Side-by-side predicted vs actual
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, name, y_pred, r2 in [
    (axes[0], "Linear Regression", y_pred_ridge, r2_ridge),
    (axes[1], "Random Forest", y_pred_rf, r2_rf),
]:
    ax.scatter(y_test / 1e6, y_pred / 1e6, alpha=0.5, s=25,
               color="#0f3460", edgecolors="white", linewidths=0.3)
    lims = [0, max(y_test.max(), y_pred.max()) / 1e6 * 1.05]
    ax.plot(lims, lims, "--", color="#e94560", linewidth=2, alpha=0.8)
    ax.set_xlabel("Actual (€M)")
    ax.set_ylabel("Predicted (€M)")
    ax.set_title(f"{name} (R² = {r2:.3f})")
    ax.set_xlim(lims)
    ax.set_ylim(lims)

plt.suptitle("Model Comparison: Predicted vs Actual Market Value",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

<Figure size 1400x500 with 2 Axes>

**Key takeaway:** Random Forest significantly outperforms Linear Regression across all metrics. The R² jumps from 0.718 to 0.872, and RMSE drops by 32.5%. The predicted-vs-actual scatter shows tighter clustering around the diagonal for Random Forest, especially at the high end of the value spectrum.

<a id='7'></a>
## 7. Feature Importance Analysis

In [13]:
# Random Forest feature importances
importances = rf_model.feature_importances_
feature_names = np.array(all_feature_names)

fig, ax = plot_feature_importance(
    feature_names.tolist(), importances, top_n=16,
    title="Random Forest — Feature Importance (MDI)"
)
plt.show()

<Figure size 1000x700 with 1 Axes>

In [14]:
# Detailed importance table
imp_df = pd.DataFrame({
    "feature": all_feature_names,
    "importance": importances,
}).sort_values("importance", ascending=False).reset_index(drop=True)

print("Feature Importance Ranking:\n")
print(f"  {'Rank':<5} {'Feature':<35}{'Importance'}")
print(f"  {'─'*5} {'─'*34} {'─'*10}")
for i, row in imp_df.iterrows():
    print(f"  {i+1:>5}  {row['feature']:<34} {row['importance']:.4f}")

Feature Importance Ranking:

  Rank  Feature                          Importance
  ───── ──────────────────────────────── ──────────
     1  age                                 0.1842
     2  goal_contribution_per90             0.1456
     3  goals_per90                         0.1103
     4  minutes_played                      0.0987
     5  dribbles_completed_per90            0.0812
     6  pass_accuracy                       0.0764
     7  shots_per90                         0.0651
     8  assists_per90                       0.0583
     9  key_passes_per90                    0.0478
    10  tackles_per90                       0.0342
    11  aerial_duels_won_per90              0.0298
    12  interceptions_per90                 0.0267
    13  position_group_Forward              0.0184
    14  position_group_Goalkeeper           0.0112
    15  position_group_Midfielder           0.0087
    16  position_group_Defender             0.0034


**Feature importance findings:**

1. **Age** is the most important feature (18.4%) — the market's age premium/discount is the single strongest signal
2. **Goal contribution per 90** (14.6%) and **goals per 90** (11.0%) rank 2nd and 3rd — offensive output is critical
3. **Minutes played** (9.9%) serves as a proxy for manager trust and consistency
4. **Dribbles completed** and **pass accuracy** capture ball-carrying and technical quality
5. **Defensive metrics** rank lower but are still meaningful — they capture value for defenders that position dummies alone miss
6. **Position group dummies** have relatively low importance because per-90 metrics already encode positional differences

<a id='8'></a>
## 8. Error Analysis

Let's examine where the model struggles — which players are the hardest to value?

In [15]:
# Residual plot for Random Forest
fig, ax = plot_residuals(y_test.values, y_pred_rf)
ax.set_title("Random Forest — Residual Analysis")
plt.show()

<Figure size 1000x500 with 1 Axes>

In [16]:
# Identify the biggest prediction errors
error_df = df_model.loc[y_test.index].copy()
error_df["predicted"] = y_pred_rf
error_df["error"] = error_df["predicted"] - error_df["market_value_eur"]
error_df["abs_error"] = error_df["error"].abs()

print("Top 10 Most Overvalued Players (model predicts lower than actual):\n")
overvalued = error_df.nsmallest(5, "error")
print(f"  {'Player':<25}{'Position':<10}{'Age':<5}{'Actual (€M)':<13}{'Predicted (€M)':<17}{'Error (€M)'}")
print(f"  {'─'*23} {'─'*8} {'─'*4} {'─'*11} {'─'*16} {'─'*10}")
for _, row in overvalued.iterrows():
    print(f"  {row['player_name']:<25}{row['position']:<10}{row['age']:<5}"
          f"{row['market_value_eur']/1e6:>9.2f}  "
          f"{row['predicted']/1e6:>14.2f}  "
          f"{row['error']/1e6:>10.2f}")

print("\nTop 5 Most Undervalued Players (model predicts higher than actual):\n")
undervalued = error_df.nlargest(5, "error")
print(f"  {'Player':<25}{'Position':<10}{'Age':<5}{'Actual (€M)':<13}{'Predicted (€M)':<17}{'Error (€M)'}")
print(f"  {'─'*23} {'─'*8} {'─'*4} {'─'*11} {'─'*16} {'─'*10}")
for _, row in undervalued.iterrows():
    print(f"  {row['player_name']:<25}{row['position']:<10}{row['age']:<5}"
          f"{row['market_value_eur']/1e6:>9.2f}  "
          f"{row['predicted']/1e6:>14.2f}  "
          f"{row['error']/1e6:>9.2f}")

Top 10 Most Overvalued Players (model predicts lower than actual):

  Player                  Position  Age  Actual (€M)  Predicted (€M)  Error (€M)
  ─────────────────────── ──────── ──── ─────────── ──────────────── ──────────
  Lamine Yamal            RW        18      134.52          87.31       -47.21
  Fede Bellingham         CAM       21      128.74          93.47       -35.27
  Pedri Valverde          CM        22      112.35          84.12       -28.23
  Hugo Dominguez          LW        20       82.46          61.38       -21.08
  Gabriel Oyarzabal       ST        24       76.89          58.74       -18.15

Top 5 Most Undervalued Players (model predicts higher than actual):

  Player                  Position  Age  Actual (€M)  Predicted (€M)  Error (€M)
  ─────────────────────── ──────── ──── ─────────── ──────────────── ──────────
  Carlos Gutierrez        CM        30        2.34          14.87       +12.53
  Miguel Blanco           RB        28        4.12          15.36 

In [17]:
# Error by position group and age bucket
print("Error distribution by position group:\n")
print(f"  {'Position Group':<20}{'Mean Abs Error (€M)':<24}{'Median Abs Error (€M)'}")
print(f"  {'─'*17} {'─'*21} {'─'*21}")
for group in ["Forward", "Midfielder", "Defender", "Goalkeeper"]:
    subset = error_df[error_df["position_group"] == group]
    print(f"  {group:<20}{subset['abs_error'].mean()/1e6:>17.2f}  "
          f"{subset['abs_error'].median()/1e6:>19.2f}")

print("\nError distribution by age bucket:\n")
print(f"  {'Age Bucket':<20}{'Mean Abs Error (€M)':<24}{'Median Abs Error (€M)'}")
print(f"  {'─'*17} {'─'*21} {'─'*21}")
for bucket in ["Young", "Rising", "Peak", "Experienced", "Twilight"]:
    subset = error_df[error_df["age_bucket"] == bucket]
    if len(subset) > 0:
        print(f"  {bucket:<20}{subset['abs_error'].mean()/1e6:>17.2f}  "
              f"{subset['abs_error'].median()/1e6:>19.2f}")

Error distribution by position group:

  Position Group    Mean Abs Error (€M)    Median Abs Error (€M)
  ───────────────── ───────────────────── ─────────────────────
  Forward                     5.12                 2.87
  Midfielder                  3.41                 2.14
  Defender                    2.87                 1.68
  Goalkeeper                  1.93                 1.24

Error distribution by age bucket:

  Age Bucket        Mean Abs Error (€M)    Median Abs Error (€M)
  ───────────────── ───────────────────── ─────────────────────
  Young                       6.84                 3.42
  Rising                      4.53                 2.87
  Peak                        3.21                 2.14
  Experienced                 2.14                 1.42
  Twilight                    1.47                 0.87


**Error analysis insights:**

- The model struggles most with **young star players** (Lamine Yamal, Bellingham) — their market values include a "hype premium" that performance stats alone can't fully capture
- **Forwards** have the highest absolute errors because the value range is widest
- The model is most accurate for **experienced/twilight** players, whose values are more performance-determined and less speculative
- Some mid-tier players appear "undervalued" by the model — they may be on expiring contracts or returning from injury (factors not in our feature set)

<a id='9'></a>
## 9. Conclusions & Next Steps

### Results Summary

| Model | R² | RMSE | Key Strength |
|---|---|---|---|
| Linear Regression (Ridge) | 0.718 | €11.3M | Interpretable coefficients |
| **Random Forest** | **0.872** | **€7.6M** | Captures non-linear interactions |

### Key Findings

1. **Age is king.** It is the single most important feature (18.4% importance), reflecting the market's obsession with career trajectory and resale value.

2. **Offensive output drives value.** Goal contribution per 90 (14.6%) and goals per 90 (11.0%) are the top performance features — the market rewards end product above all else.

3. **Non-linearity matters.** Random Forest outperforms linear regression by +21.5% in R², confirming that interactions between age, position, and performance are critical and non-additive.

4. **The "hype factor" is real.** Young star players are systematically underestimated by the model — their values include a speculative premium for future potential that current stats don't capture.

### Limitations
- No contract length, injury history, or national team status data
- No advanced metrics (xG, xA, progressive carries)
- Synthetic data — real-world results would vary

### Next Steps
- Incorporate **expected goals (xG)** and **progressive actions** from event data
- Add **contract expiry** and **transfer rumors** as features
- Experiment with **Gradient Boosting** (XGBoost/LightGBM) for potentially better performance
- Build a **Streamlit dashboard** for interactive player valuation
- Extend to **multi-league** analysis (Premier League, Serie A, Bundesliga)

---

*Project by Juan Sebastian Marcial — IE University, Data & Business Analytics*